# Preparation

## Setup

In [ ]:
!pip install --upgrade transformers

In [ ]:
!pip install datasets

In [ ]:
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
!pip install pyphen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.8 MB/s eta 0:00:00


## LoRA

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

def applyLoRA(model):
  # LoRA configuration
  lora_config = LoraConfig(
      r=8,                # rank of LoRA matrices
      lora_alpha=32,      # scaling factor
      target_modules=["c_attn"],  # GPT2 attention layers to adapt
      lora_dropout=0.1,
      bias="none",
      task_type=TaskType.CAUSAL_LM,
  )
  # Apply LoRA to model
  return get_peft_model(model, lora_config)

## GPT2 small

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import DataCollatorForLanguageModeling

# Load GPT2 small and tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Add pad token to GPT2, to deal with text of varying length
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

# Apply LoRA to the model
model = applyLoRA(model)

# Prepare data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False) # GPT won't use mask tokens

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:1768: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## Dataset conversion

### Modification

In [ ]:
!wget -nc --header="User-Agent: Mozilla/5.0" https://data.mendeley.com/public-files/datasets/3t9vbwxgr5/files/d2c58546-d8be-4d57-af14-b61337c927b2/file_downloaded -O "data.csv"

--2025-06-12 10:28:37--  https://data.mendeley.com/public-files/datasets/3t9vbwxgr5/files/d2c58546-d8be-4d57-af14-b61337c927b2/file_downloaded
Resolving data.mendeley.com (data.mendeley.com)... 162.159.133.86, 162.159.130.86
Connecting to data.mendeley.com (data.mendeley.com)|162.159.133.86|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/036e1862-4d9b-4ea8-b762-7ae3282ba2c3 [following]
--2025-06-12 10:28:37--  https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/036e1862-4d9b-4ea8-b762-7ae3282ba2c3
Resolving prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com (prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com)... 52.218.116.90, 3.5.72.20, 52.92.16.58, ...
Connecting to prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com (prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com)|52.218.116.90|:443..

In [ ]:
import pandas as pd

# Load CSV
raw_df = pd.read_csv("data.csv")
print("All genres:", raw_df['genre'].unique())

# Leave only one genre
global GENRE
GENRE = 'blues'
raw_df = raw_df[raw_df['genre'] == GENRE] # reduce the dataset size
raw_df = raw_df[raw_df['obscene'] < 0.1] # reduce the amount of vulgar words
print(f"The most rude lyric in the resulting dataset: ", raw_df.loc[raw_df['obscene'].idxmax()]["lyrics"])
print(f"Amount of rows for {GENRE}:", raw_df.shape[0])

# Pick all keywords first
df = raw_df.iloc[:,7:-2]
print("\nOld keywords:\n")
print(df.max())

# Clear the data
df = df.drop(columns=['loudness', 'acousticness', 'instrumentalness', 'communication', 'music']) # not descriptive for lyrics
df = df.drop(columns=['shake the audience', 'like/girls', 'family/gospel', 'family/spiritual', 'light/visual perceptions', 'movement/places', 'dating']) # will not provide enough examples (max value too low)
df = df.drop(columns=['obscene']) # obscene lyrics are filtered out

# Convert keywords to their adjective form for prompting
df = df.rename(columns={'valence': 'happy'})  # the higher values for valence mean uplifting
df = df.rename(columns={'energy': 'energetic'})
df = df.rename(columns={'danceability': 'danceable'})
df = df.rename(columns={'violence': 'violent'})
df = df.rename(columns={'sadness': 'sad'})
df = df.rename(columns={'feelings': 'emotional'})
df = df.rename(columns={'world/life': 'timeless'})
df = df.rename(columns={'night/time': 'temporal'})  # means related to time
print("\nUpdated keywords:\n")
print(df.max())

# Add topic and lyrics
df['topic'] = raw_df['topic']
df['lyrics'] = raw_df['lyrics']
print("\nFinal columns:", df.columns.to_list())

All genres: ['pop' 'country' 'blues' 'jazz' 'reggae' 'rock' 'hip hop']
The most rude lyric in the resulting dataset:  give worry come depend give lie believe true wrap arm street help wonder know go talk know feel realize better stop tear apart better stop break heart whoa better stop time time try walk away easy soul tear resign leave better stop tear apart better stop break heart whoa better stop better stop time sorry forever believe walk better stop tear apart better stop break lonely heart whoa whoa better whoa whoa whoa better stop better stop
Amount of rows for blues: 3632

Old keywords:

dating                      0.647706
violence                    0.970395
world/life                  0.852695
night/time                  0.852394
shake the audience          0.402632
family/gospel               0.502873
romantic                    0.864662
communication               0.613830
obscene                     0.099887
music                       0.901610
movement/places            

### Get values necessary for data processing

In [ ]:
# Get keyword strings
global KEYWORDS
KEYWORDS = df.columns.tolist()[:-2]

# Get topics
global TOPICS
TOPICS = df['topic'].unique()

# Return topcis in their noun form for prompting
def get_topic_noun(topic):
  if topic == "obscene":
    return "an obscene topic"
  elif topic == "romantic":
    return "a romantic topic"
  return topic

print("All keywords:",KEYWORDS)
print("All topics:",TOPICS)
print("Topic noun examples:",get_topic_noun(TOPICS[2]),",",get_topic_noun(TOPICS[0]))


All keywords: ['violent', 'timeless', 'temporal', 'romantic', 'sad', 'emotional', 'danceable', 'happy', 'energetic']
All topics: ['music' 'violence' 'world/life' 'sadness' 'romantic' 'feelings'
 'night/time']
Topic noun examples: world/life , music


### Convert data into a prompt for GPT

In [ ]:
from datasets import Dataset

# GPT is trained with text in some specific format
def make_prompt(row):
    # Convert topic
    topic = get_topic_noun(row['topic'])

    # Pick keywords with values > 0.5 or a keyword with a highest value to add more consistency in the prompt
    attributes = []
    for keyword in KEYWORDS:
      if(float(row[keyword]) > 0.5):
        attributes.append(keyword)
    if(len(attributes) == 0):
      attributes.append("generic")
    attributes = " and ".join(attributes) # convert to string

    # Generate a prompt per line
    return f"Generate a line for a {GENRE} song about {topic} with {attributes} feel.\n{row['lyrics']}\n"

# Example
print("Prompt made of the arbitrary row in the dataset")
print(make_prompt(df.iloc[178]))

Prompt made of the arbitrary row in the dataset
Generate a line for a blues song about a romantic topic with generic feel.
forever darling true forever promise darling return soul dear forever burn heart command dear hold make desire dear keep goal forever rest days love ways promise darling return soul dear forever burn forever rest days love ways promise darling return soul dear forever burn



# Training

## Load the dataset

### Create dataset for GPT

In [ ]:
all_prompts = []

# Generite prompts for every "line" of lyrics row by row
for _, row in df.iterrows():
    all_prompts.append({"text": make_prompt(row)})  # wrap each prompt in a dict for Hugging Face Dataset
print(all_prompts[596])

# Create dataset for GPT
dataset = Dataset.from_list(all_prompts)
print(dataset[:1])

{'text': 'Generate a line for a blues song about world/life with generic feel.\nwant play baby time expensive little play heart make furious want baby girl know tell like ashamed conscience guide tell like believe forget foolish pride life short sorrow today go tomorrow want live baby live tellin like play baby tellin like cause good\n'}
{'text': ['Generate a line for a blues song about music with danceable and happy and energetic feel.\nthink know songs write starcrossed sell harmonies go tune time silver time blue midnight dancers spin silver moon stone romancers answer time silver time blue instrumental flute electric slide guitar time silver time blue think know songs write time silver time blue instrumental flute electric slide guitar time silver time blue\n']}


### Initialize the tokenized dataset

In [ ]:
from datasets import load_dataset

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

def load_tokenized_dataset():
  return dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Load the dataset
tokenized_dataset = load_tokenized_dataset()

Map:   0%|          | 0/3632 [00:00<?, ? examples/s]

## Prepare training

### Google Drive mount

In [ ]:
from google.colab import drive

# Connect my Google Drive to save things
drive.mount('/content/drive')

# Save model checkpoints
global SAVE_PATH
SAVE_PATH = "./drive/MyDrive/Uni/gpt2-lora-train"

Mounted at /content/drive


### Check if GPU is used

In [ ]:
import torch
print(torch.cuda.is_available())  # Should print True if GPU is enabled
if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))  # GPU model name

True
Tesla T4


In [ ]:
!nvidia-smi

Wed Jun 11 16:40:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Define training

In [ ]:
from transformers import Trainer, TrainingArguments

# Training args
training_args = TrainingArguments(
    weight_decay=0.01,                # Prevent overfitting
    output_dir=SAVE_PATH,             # Save directory
    num_train_epochs=6,               # Reasonable for small dataset
    per_device_train_batch_size=4,    # Lower batch size = fits in memory
    gradient_accumulation_steps=4,    # Simulates larger batch (4x4=16)
    logging_steps=1,                  # Print loss every step
    save_steps=1,                     # Save checkpoint every step
    save_total_limit=1,               # Keep only 1 checkpoint
    learning_rate=5e-5,               # Good starting point
    fp16=False,                       # Set True if using supported GPU
    report_to="none",                 # Avoid W&B prompts
    overwrite_output_dir=True,        # Replace previous training run
)
training_args.evaluation_strategy = "steps" # set "steps" or "epoch" if you have validation

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


## Train

In [ ]:
chk_pnt = SAVE_PATH + "/checkpoint-1362"

# Training
trainer.train(resume_from_checkpoint=chk_pnt)

Didn't manage to set back the RNG states of the CUDA because of the following error:
 'cuda'
This won't yield the same results as if the training had not been interrupted.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
706,4.044900
707,4.014400
708,3.970300
709,4.698500
710,4.373100
711,4.298400
712,4.463600
713,3.597800
714,4.007900
715,3.914000


/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:4064: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to numerical ordering")
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:4064: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to numerical ordering")
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:4064: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to numerical ordering")
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:4064: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable o

TrainOutput(global_step=1362, training_loss=1.9885664085984756, metrics={'train_runtime': 622.7491, 'train_samples_per_second': 34.993, 'train_steps_per_second': 2.187, 'total_flos': 1381480302329856.0, 'train_loss': 1.9885664085984756, 'epoch': 6.0})

## Save a checkpoint to GDrive shared folder (update the checkpoint number in the code below!)

In [ ]:
!cp -r /content/drive/MyDrive/Uni/gpt2-lora-train/checkpoint-1362 /content/drive/MyDrive/Uni/gpt2-lora/checkpoint-1362

## Load checkoint to mounted GDrive

In [ ]:
# First define load_checkpoint
# tokenizer, model = load_checkpoint(1362, True)

# Usage

## Prompt for inference

In [ ]:
import random

def make_inference_prompt(topic_index, keyword_indexes):
  input = {}

  # Set topic
  input["topic"] = TOPICS[topic_index]

  # Set keywords
  for keyword in KEYWORDS:
    input[keyword] = 0.00
  for keyword_index in keyword_indexes:
    input[KEYWORDS[keyword_index]] = 1

  # Set lyrics
  input["lyrics"] = "."

  # Create prompt
  return make_prompt(input)[:-2]

# Example
topic = 4
keywords = [5,8]
print("Topic:",topic,"-",TOPICS[topic])
print("Keywords:",[(lambda x: str(x)+" - "+KEYWORDS[x])(x) for x in keywords])
print("\nInference prompt:")
print(make_inference_prompt(topic,keywords))

Topic: 4 - romantic
Keywords: ['5 - emotional', '8 - energetic']

Inference prompt:
Generate a line for a blues song about a romantic topic with emotional and energetic feel.



## Load the model from a checkpoint

In [ ]:
!gdown --folder https://drive.google.com/drive/folders/16uebnCWNHAiQJMbTpj70bRY9dAbnPh6m

Retrieving folder contents
Retrieving folder 14WSG53aqbjEhv7VM7m0hPSRE_4sFWY2L checkpoint-1362
Processing file 1TUa8VXyDMgXayTbPznxEY6scB7AN1NIY adapter_config.json
Processing file 1c8Ek2gMTHwkzZYEJjunwhtlW0oH3GQPy adapter_model.safetensors
Processing file 1TVAeY6MRWCuS2GLdoUM-1A-L0SE1EXOo merges.txt
Processing file 1DgOHkoszkQbJffNm99rzbSnPdngoldNJ optimizer.pt
Processing file 1xdBTYAvNKnJcXoF9EKvSnk94tf7WZvdP README.md
Processing file 1ZtiQOjcE-ne5Xh10d6IlP04xQpVGxB-h rng_state.pth
Processing file 1VMfPNyBSSp8wqGIeZzti6HELiWi6RYPV scheduler.pt
Processing file 12YPJ09vRZAMLn1VMU4aN14fNXt1AMAtF special_tokens_map.json
Processing file 1cRd38fhrMxvW6P4F8upZya08rLXyA3oz tokenizer_config.json
Processing file 10IbSwc5CQEMQHZDyYTKhkVb2-JH6L7lO trainer_state.json
Processing file 11sRdbsf7eZluXsjIs9H4Yuf3Pv1qxIyd training_args.bin
Processing file 1mdOdhs2Oz78YHHMPlcuN5Wq7K1EPZJS3 vocab.json
Retrieving folder contents completed
Building directory structure
Building directory structure completed

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

def load_checkpoint(num, train=False):
  # Load tokenizer (same as original model)
  tokenizer = AutoTokenizer.from_pretrained("gpt2")

  # Load LoRA config from checkpoint
  if train:
    peft_model_path = SAVE_PATH + "/checkpoint-" + str(num) # for mounted GDrive
  else:
    peft_model_path = "gpt2-lora/checkpoint-" + str(num)

  config = PeftConfig.from_pretrained(peft_model_path)

  # Load base model with LoRA adapter
  base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
  model = PeftModel.from_pretrained(base_model, peft_model_path)
  return tokenizer, model

In [ ]:
tokenizer, model = load_checkpoint(1362)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Generate lyrics line

In [ ]:
# Inference
def infer_lyrics(prompt):
  inputs = tokenizer(prompt, return_tensors="pt")
  outputs = model.generate(
      **inputs,
      max_length=300,
      temperature=0.7,           # adds some creativity, but not too random
      top_k=50,                  # restricts to top-k likely tokens
      top_p=0.9,                 # nucleus sampling: focus on top 90% prob mass
      do_sample=True,            # enables sampling
      num_return_sequences=1,    # only 1 line
      repetition_penalty=1.1,    # try to avoid endless repetition
  )
  return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Example
# topic = 4
# keywords = [5]
# print("Topic:",topic,"-",TOPICS[topic])
# print("Keywords:",[(lambda x: str(x)+" - "+KEYWORDS[x])(x) for x in keywords])

# example_prompt = make_inference_prompt(topic, keywords)
# print("\nExample prompt:")
# print(example_prompt)

# print("Generated lyrics line:")
# print(infer_lyrics(example_prompt))

## Rhyming

### Helpers

In [ ]:
!pip install syllapy

In [ ]:
import syllapy

# Get first 8 (by default) syllables
def top_line(text, syllables=8):
    words = text.split()
    result = []
    syllable_count = 0

    for word in words:
        syls = syllapy.count(word)
        if syllable_count + syls > syllables:
            break
        result.append(word)
        syllable_count += syls

    return ' '.join(result)

# Remove substring and everything before it
def remove_substring(substring, text):
    index = text.find(substring)
    if index != -1:
        return text[index + len(substring):]
    return text  # return original if substring not found

### Rhyme searching

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
!pip install g2p_en

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.8 MB/s eta 0:00:00
  Created wheel for distance: filename=Distance-0.1.3-py3-none-any.whl size=16256 sha256=fec3c6c959a549a0c934ec59d7992c720bad0fda16fd75e275db2a9d77b01109
  Stored in directory: /root/.cache/pip/wheels/fb/cd/9c/3ab5d666e3bcacc58900b10959edd3816cc9557c7337986322
Successfully built distance


In [ ]:
from g2p_en import G2p
import re
from collections import defaultdict

# Initialize G2p converter
g2p_converter = G2p()

# Phonetic rhyming logic
def get_phonetic_rhyming_part(phonemes):

    # Extract the 'rhyming part' from a list of ARPABET phonemes.
    vowel_pattern = re.compile(r'[AEIOU][0-2]?')

    for i in reversed(range(len(phonemes))):
        if vowel_pattern.match(phonemes[i]):
            return phonemes[i:]
    return []

# Check if two words approximately rhyme based on their G2P phonetic similarity.
def are_approx_rhymes(word1, word2, min_shared_phonemes=2):
    if word1.lower() == word2.lower(): # A word doesn't rhyme with itself
        return False

    phonemes1 = g2p_converter(word1.lower())
    phonemes2 = g2p_converter(word2.lower())

    if not phonemes1 or not phonemes2:
        return False

    rhyming_part1 = get_phonetic_rhyming_part(phonemes1)
    rhyming_part2 = get_phonetic_rhyming_part(phonemes2)

    if not rhyming_part1 or not rhyming_part2:
        return False

    min_len = min(len(rhyming_part1), len(rhyming_part2))

    shared_count = 0
    for i in range(1, min_len + 1):
        p1 = rhyming_part1[-i].rstrip('012')
        p2 = rhyming_part2[-i].rstrip('012')

        if p1 == p2:
            shared_count += 1
        else:
            break

    return shared_count >= min_shared_phonemes

def find_rhymes(supplied_word, text, min_shared_phonemes=2):
    text = remove_substring(top_line(text, syllables=7), text) # ignore first 7 syllables

    # Pre-process the supplied word's phonetics once
    supplied_word_phonemes = g2p_converter(supplied_word.lower())
    if not supplied_word_phonemes:
        return []

    # Clean and tokenize the text
    cleaned_text = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    words_in_text = nltk.word_tokenize(cleaned_text)

    # Collect unique words in their original order
    seen = set()
    unique_words_in_order = []
    for w in words_in_text:
        if len(w) > 1 and w != supplied_word.lower() and w not in seen:
            seen.add(w)
            unique_words_in_order.append(w)

    found_rhymes = []

    for word_from_text in unique_words_in_order:
        if are_approx_rhymes(supplied_word, word_from_text, min_shared_phonemes):
            found_rhymes.append(word_from_text)

    return found_rhymes


# Example
sample_text = "The old gray cat sat by the mat. He wore a fine hat, a real treat."
example_search = "cat"
print("Sample text:", sample_text)
print("Search word:", example_search)
print(f"Rhymes for 'cat' (first 7 syllables ignored): {find_rhymes(example_search, sample_text, min_shared_phonemes=2)}")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


Sample text: The old gray cat sat by the mat. He wore a fine hat, a real treat.
Search word: cat
Rhymes for 'cat' (first 7 syllables ignored): ['mat', 'hat']


### Get line with target word

In [ ]:
import syllapy

def get_syllables(text):
    words = text.split()
    syllables = []
    for word in words:
        count = syllapy.count(word)
        syllables.append((word, count))
    return syllables

def find_line(target_word, text, syllable_count=8):
    syllables = get_syllables(text)
    results = []

    # Walk through all positions
    for i in range(len(syllables)):
        word, count = syllables[i]

        if word.lower() == target_word.lower():
            # Walk backwards to collect n syllables
            total = 0
            j = i
            chunk_words = []

            while j >= 0 and total < syllable_count:
                w, c = syllables[j]
                chunk_words.insert(0, w)
                total += c
                j -= 1

            if total == syllable_count:
                results.append(' '.join(chunk_words))

    return results

# Example usage:
example_text = "I wandered lonely as a cloud that floats on high o'er vales and hills"
target_word = "vales"
print("Sample text:", example_text)
print("Target word:", target_word)
print(f"Extracted line: {find_line(target_word, example_text)}")

Sample text: I wandered lonely as a cloud that floats on high o'er vales and hills
Target word: vales
Extracted line: ["that floats on high o'er vales"]


## Censor profanity

In [ ]:
!pip install better_profanity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
from better_profanity import profanity

# Initialize
profanity.load_censor_words()

# Final result

In [ ]:
# Show choosable values
def lyrics_info():
  print("Topics:\n")
  for i, item in enumerate(TOPICS):
      print(f"{i}: {item}")
  print("\nKeywords:\n")
  for i, item in enumerate(KEYWORDS):
      print(f"{i}: {item}")
  print("\n")

# Primitive way to make rhyming lyrics
def generate_lyrics_chunk(topic_index, keyword_indexes):
  gpt_prompt = make_inference_prompt(topic, keywords)
  gpt_line = infer_lyrics(gpt_prompt)
  gpt_line = "\n".join(gpt_line.split("\n")[1:]) # remove the prompt part

  # Prepare the first line
  line = top_line(gpt_line) # take first 8 syllables
  last_word = line.strip().split()[-1] # take the last word to rhyme

  final_lines = [line.capitalize()+","]

  # Generate rhyming lines
  for i in range(0,3):
    # Generate a new rhyming line
    rhymes = []
    while len(rhymes) == 0:
      gpt_line = infer_lyrics(gpt_prompt)
      gpt_line = "\n".join(gpt_line.split("\n")[1:]) # remove the prompt part
      print(i,"line:",gpt_line)
      rhymes = find_rhymes(last_word, gpt_line, 1)
      print(i,"rhymes:",rhymes)

    # Update the variables
    rhyme_lines = []
    rhyme_index = 0
    while len(rhyme_lines) == 0:
      rhyme_lines = find_line(rhymes[rhyme_index], gpt_line)
      rhyme_index = rhyme_index + 1
    rhyme_line = rhyme_lines[0] # take the first result only
    print(rhyme_line)
    last_word = rhyme_line.strip().split()[-1]
    final_lines.append(rhyme_line.capitalize()+",")

  return profanity.censor("\n".join(final_lines))

# Example
lyrics_info()
topic = 4
keywords = [5]
print("Chosen topic:",topic,"-",TOPICS[topic])
print("Chosen keywords:",[(lambda x: str(x)+" - "+KEYWORDS[x])(x) for x in keywords],"\n")

generated_lines = generate_lyrics_chunk(topic, keywords)
print("\nGenerated lines:")
print(generated_lines)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Topics:

0: music
1: violence
2: world/life
3: sadness
4: romantic
5: feelings
6: night/time

Keywords:

0: violent
1: timeless
2: temporal
3: romantic
4: sad
5: emotional
6: danceable
7: happy
8: energetic


Chosen topic: 4 - romantic
Chosen keywords: ['5 - emotional'] 



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


0 line: live life live day leave time heart talk baby know say goodbye come home go away world peace make sense love cause happy face look like lover think want break things walk down lonely stand out tell your soul hear change see light forget thing wrong do somethin get back lose matter fade mind give way hope don't stay quiet wait keep going watch long turn blind eyes stop move night slow start die fast drive past bend point need hide lie hand eye open mouth hold off right side let head left shoulder arm elbow sit leg up knee bent knees neck straight shoulders front hips backward body meet chest height elbows spread arms wide legs reach forward waist lower belly lean thighs pull tight upper thigh extend hands deep sides inches apart feet behind foot kneel ankles hang spine inch butt toes push breast into hip raise heels weight hard clean fingertips nails hair brush teeth nail mark fingers cut lip take cheek lips nose finger tip tongue touch chin lift toe skin fill smile burn cheeks 

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


1 line: dream come go leave like home wake stay want sleep think forget look out face dreamcome fall goodbye know say farewell day promise keep time long stop remember see away past life let back tear night smile hope tears lie down heart walk into heaven tell goodnight world thank god God help you don't cry just hide eye blind stare eyes forever hear sound listen tonight turn blue hearts right start love lose kiss hold awake whisper take little finger lick softly talk show baby make head laugh scream yell louder live sing tune wait till morning oh fool cause there's gonna be breakin' tomorrow stand up give me your hands hand free arm grab tighty slip open mouth speak voice breathe hard breath try to bend knees cradle pull on toe touch toes inch fingers sit shoulder lean shoulders lay stomach spread legs wide apart neck tighten arms tie feet bare ankles wrap waist around hips hang belly sway thighs extend elbows lower body over chest knee deep throat warm lips sweetheart kisses close n

# Evaluation

## Raw pretrained GPT small

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Load GPT2 small and tokenizer
model_name = "gpt2"
raw_tokenizer = GPT2Tokenizer.from_pretrained(model_name)
raw_model = GPT2LMHeadModel.from_pretrained(model_name)

In [ ]:
def raw_infer_lyrics(prompt):
  # Tokenize input
  input_ids = raw_tokenizer.encode(prompt, return_tensors='pt')

  # Generate continuation (same settings as for the fine tuned model)
  output = raw_model.generate(
      input_ids,
      max_length=300,
      temperature=0.7,
      top_k=50,
      top_p=0.9,
      do_sample=True,
      num_return_sequences=1,
      repetition_penalty=1.1,
  )

  # Decode
  the_final_result = tokenizer.decode(output[0], skip_special_tokens=True)
  return "\n".join(the_final_result.split("\n")[1:]) # remove the prompt from the result

# Example
lyrics_info()
topic = 4
keywords = [5]
print("Chosen topic:",topic,"-",TOPICS[topic])
print("Chosen keywords:",[(lambda x: str(x)+" - "+KEYWORDS[x])(x) for x in keywords],"\n")

new_prompt = make_inference_prompt(topic, keywords)
print("The prompt: ", new_prompt)
print("\nGenerated lines:")
print(raw_infer_lyrics(new_prompt))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Topics:

0: music
1: violence
2: world/life
3: sadness
4: romantic
5: feelings
6: night/time

Keywords:

0: violent
1: timeless
2: temporal
3: romantic
4: sad
5: emotional
6: danceable
7: happy
8: energetic


Chosen topic: 4 - romantic
Chosen keywords: ['5 - emotional'] 

The prompt:  Generate a line for a blues song about a romantic topic with emotional feel.


Generated lines:
-Use the "play" button to change all lyrics and replace them in your music as well.

 - Use this tool when you have problems finding what songs are currently playing on Spotify, or if someone has an album which is not available yet (so we can't play it). If there's one thing that will help us find better ways of looking at our Music files than just using their page numbers then please give feedback!


## 10 test cases

In [ ]:
# Test case
def run_test_case(topic_index, keyword_indexes):
  print("\n\n--- TEST CASE ---")
  print("Chosen topic:",topic_index,"-",TOPICS[topic])
  print("Chosen keywords:",[(lambda x: str(x)+" - "+KEYWORDS[x])(x) for x in keyword_indexes],"\n")

  raw_gpt_prompt = make_inference_prompt(topic_index, keyword_indexes)
  print("The prompt:", raw_gpt_prompt)

  print("\nGPT SMALL RAW:")
  print(raw_infer_lyrics(raw_gpt_prompt),"\n")

  generated_lines = generate_lyrics_chunk(topic_index, keyword_indexes)
  print("\nFINE-TUNED GPT:")
  print(generated_lines)

test_cases = [
    {'topic': 0, 'keywords': [6, 8]},          # music + danceable, energetic
    {'topic': 1, 'keywords': [0]},             # violence + violent
    {'topic': 2, 'keywords': [1, 2]},          # world/life + timeless, temporal
    {'topic': 3, 'keywords': [4, 5]},          # sadness + sad, emotional
    {'topic': 4, 'keywords': [3, 5]},          # romantic + romantic, emotional
    {'topic': 5, 'keywords': [5, 7]},          # feelings + emotional, happy
    {'topic': 6, 'keywords': [1]},             # night/time + timeless
    {'topic': 0, 'keywords': [8]},             # music + energetic
    {'topic': 3, 'keywords': [4]},             # sadness + sad
    {'topic': 2, 'keywords': [1, 7, 8]}         # world/life + timeless, happy, energetic
]

lyrics_info()
for test_case in test_cases:
  run_test_case(test_case['topic'],test_case['keywords'])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Topics:

0: music
1: violence
2: world/life
3: sadness
4: romantic
5: feelings
6: night/time

Keywords:

0: violent
1: timeless
2: temporal
3: romantic
4: sad
5: emotional
6: danceable
7: happy
8: energetic




--- TEST CASE ---
Chosen topic: 0 - romantic
Chosen keywords: ['6 - danceable', '8 - energetic'] 

The prompt: Generate a line for a blues song about music with danceable and energetic feel.


GPT SMALL RAW:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Lyrics: I've been living in this town since 1994 when the first record was released, but not yet that many people know of it so maybe you should check out my book on how to make something really cool as well!

[From "Boomerang" by Brian May): This is what happens at night like we call our home life where there are no lights or anything other than light bulbs around... [You will hear] sounds from all over your house coming through some part of the building which makes me want to scream because then they're going to turn off their TV screens while watching us play video games.... The sound has only just begun..... So if anyone had any idea why i love making these songs please share them here - http://www2freethepinkmusic.com/tunes#sthash_gZRQHWJzkUjM8GmEwC3VlcGFjaWR1YTpbml0FDEyMTKvbXNlbnRyIxNTcS6nRoZXOu5ODhMC4ND9oNWiOG==[/quote] 



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

This is not the first time we have created this project, so be sure to check it out! We hope you enjoy our work as much as us…and please share your thoughts in comments below or on social media using #myknees 



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


0 line: love know remember love hear tell heart leave tear come tears go break time fall talk think wrong let forget wish say goodbye give look like forgive blame stay lose smile show hope cause fade away see dream take hold turn back bend right side wanna be lonely good night just wait till dawn tonight cry kiss sleep dreams fill want better nights soul lie hate ask believe wonder why not do things more difficult than life hurt mind try harder hide face grow worse still doubt scream listen fear stand blind eye watch eyes dark thoughts stare out of focus worry breathe hard swallow don't follow down pain make world burn alive sing songs louder now live longer keep regret long past have reason to question need stronger resolve finally bring shame walk home baby girl call stop mother crying wake up sound asleep speak words so tired you're sick all morning light blue rain sky black cloud start bright white lightning storm spring cold sun slow wind head deep breath drop shine in soft breeze

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


1 line: play time light play day life leave forget take change come home break go away stay remember stand out stop say goodbye lie look wait let think hold tell want know like live tonight night talk do better mind turn right walk need sleep see tomorrow morning dream dreams don't believe no hope keep fight promise start hide try hard get heart long stick hurt love give chase drive lose tear drop tears fall deep hear sing music soul whisper kiss beat listen eye wake smile fade eyes breathe breath heavy draw hearts fill your world wish never be lonely moment hand free spirit touch ring fingers open hands dance tune till dawn burn shine bright moon brighter sky white color glow dark gray suns grow cold blue red yellow green baby face cry make happy little boy scream ask father thank god cradle bless kids old brother crying daddy send good word call mother mom miss family friend help son bring back friends step past daughter share cause show kid run errands watch parent spend days child 

## Rhyme density


Basically 100%

## Raw vs fine-tuned

In [ ]:
"""
--- TEST CASE ---
Chosen topic: 0 - romantic
Chosen keywords: ['6 - danceable', '8 - energetic']

The prompt: Generate a line for a blues song about music with danceable and energetic feel.


GPT SMALL RAW:
Lyrics: I've been living in this town since 1994 when the first record was released, but not yet that many people know of it so maybe you should check out my book on how to make something really cool as well!

[From "Boomerang" by Brian May): This is what happens at night like we call our home life where there are no lights or anything other than light bulbs around... [You will hear] sounds from all over your house coming through some part of the building which makes me want to scream because then they're going to turn off their TV screens while watching us play video games.... The sound has only just begun..... So if anyone had any idea why i love making these songs please share them here - http://www2freethepinkmusic.com/tunes#sthash_gZRQHWJzkUjM8GmEwC3VlcGFjaWR1YTpbml0FDEyMTKvbXNlbnRyIxNTcS6nRoZXOu5ODhMC4ND9oNWiOG==[/quote]


FINE-TUNED GPT:
Like like tell give go know come let,
Sky go green white baby girl heart,
Go stand hear wonder leave stare night,
Stand come back home look say goodbye wait,


--- TEST CASE ---
Chosen topic: 1 - romantic
Chosen keywords: ['0 - violent']

The prompt: Generate a line for a blues song about violence with violent feel.


GPT SMALL RAW:
I think we're on to something here," she says. "The world is changing, and it's going up against the fact that people are starting doing things differently."


FINE-TUNED GPT:
Come come stay away cry go tell,
Good goodbye nice big happy little,
Night time walk sleep come play talk tell,
Get somethin' come fall time ****,


--- TEST CASE ---
Chosen topic: 2 - romantic
Chosen keywords: ['1 - timeless', '2 - temporal']

The prompt: Generate a line for a blues song about world/life with timeless and temporal feel.


GPT SMALL RAW:
The title of the project comes from an email sent to me by "Vladimir" Eustace, who is writing his first solo album as well as several songs that will be released in 2017 under IRL name: The First Five Songs (with all five albums being single-album singles). He also wrote this book on how he had come up with it during summer 2014 when playing guitar at University College London's Rockin' In Locker Studios was something like 20 hours long - or maybe even longer! This has been my personal favorite part because while not every day you're going to get some great music out of your life just so someone else can play those tunes together without any hassle whatsoever but if one person are able to make such awesome work then they have made their own rock n roll record which makes no difference what genre these records were produced within :)


FINE-TUNED GPT:
Love come love fall look see eye tell,
Like love stay hold leave mind wait till,
Love kiss goodbye hug heart walk leave soul,
Sleep know give say come out stay tell,


--- TEST CASE ---
Chosen topic: 3 - romantic
Chosen keywords: ['4 - sad', '5 - emotional']

The prompt: Generate a line for a blues song about sadness with sad and emotional feel.


GPT SMALL RAW:
If you're having trouble getting the message out, try this one: "I'm sorry… I don't want to hear that sound in your head." You'll get lots of other fun ideas like these at home!


FINE-TUNED GPT:
Good morning heart good night smile hand,
Give remember say life keep hold,
Rain drop shine fall sunlight touch wind,
Think say goodbye love stop pain talk mind,


--- TEST CASE ---
Chosen topic: 4 - romantic
Chosen keywords: ['3 - romantic', '5 - emotional']

The prompt: Generate a line for a blues song about a romantic topic with romantic and emotional feel.


GPT SMALL RAW:
A string of phrases can be played to describe the mood, tone or feeling on an individual's body (see "How Do I Know This?"). The phrase is used often in songs such as Goodnight Blues & Stompin' In My Pants - that same music was featured by Drums Of Love during their first tour together! A list of all known examples could then have been found from various sources including Wikipedia , here are some notable ones:

 "I Am Your Man". When asked what he thought they should say if someone had called him names like 'My Boyfriend', it has become common knowledge when people call you his name because this does not necessarily mean any specific person but rather one who refers to himself/herself only indirectly." [1]                             https://www .youtube-nocookieurls / youtube_nscntemplates ?v=3C6JFZ5X4bM8Kw9gDkNjR2PQfHt0q7mWxGY&showplayerid=1825475055


FINE-TUNED GPT:
Lose stay lose sleep talk like leave kiss,
Just try again next year sweet miss,
Come go get baby see look face,
Remember all time happy dance,


--- TEST CASE ---
Chosen topic: 5 - romantic
Chosen keywords: ['5 - emotional', '7 - happy']

The prompt: Generate a line for a blues song about feelings with emotional and happy feel.


GPT SMALL RAW:
In the U.S., it's used to express how depressed or anxious you are, but if your feeling is like that then use this as an opportunity to talk out of turn in conversation instead. This can be done by using something called "love letter" which has two parts: one part on writing love letters (the other being asking people not to write them) so they don't get angry at eachother; another part explaining why what happened last night wasn'nt really anything special! The reason I do these things though? It makes me want more friends than just talking through my depression - because most others will know exactly who wrote those words when we first met during our honeymoon period!! We all have stories too... Read More


FINE-TUNED GPT:
Blue sky blue heaven earth soul,
Go up come out stay down look tell,
Go want come right away hear soul,
Cause cause wait long come die lie fall,


--- TEST CASE ---
Chosen topic: 6 - romantic
Chosen keywords: ['1 - timeless']

The prompt: Generate a line for a blues song about night/time with timeless feel.


GPT SMALL RAW:
I know you don't think I'm going to be able take this one, but if not then just give it another try! You might even love the rest of my work!!

FINE-TUNED GPT:
Cold come get cold go give way blow,
Want see wonder forget dream know,
Dreamin dreamnin dreams mind heart go,
Hope dream wish make sure tomorrow,


--- TEST CASE ---
Chosen topic: 0 - romantic
Chosen keywords: ['8 - energetic']

The prompt: Generate a line for a blues song about music with energetic feel.


GPT SMALL RAW:
Tired of playing it on your phone? Then you've come to the right place! We have tons of free downloadable songs and other cool content in our app, including new ones like "My Little Pony" (featuring Diddy Kong), "Mountain Dew," or even The Music Store's FREE Rave Mixtape as well... but first let me give an extra shout out at all those awesome artists who are making this project possible:

-- Michael O'Brien- @michaelo_obrien@gmail .com


FINE-TUNED GPT:
Light dance light look go come out shine,
Cry call do die young tears like pain,
Go to sleep baby break open,
Stay long walk lonely night listen,


--- TEST CASE ---
Chosen topic: 3 - romantic
Chosen keywords: ['4 - sad']

The prompt: Generate a line for a blues song about sadness with sad feel.


GPT SMALL RAW:
I know I have done that before, but it seems to me like the "Sadness" section is so big and too long in my head (and really difficult) as well. It's definitely not fair or helpful if you're making something fun of your music when they just make up this part entirely from scratch!

 Here are some things which will help: - Get creative : You'll need an editor such at least one piecewise shape/shape tool out there where people can create their own shapes based on what looks best together... Or even more importantly try different kinds of sounds over time using whatever tools available without having any preconceived notions attached..... Be sure to get good feedback first by letting others hear who does better than yourself; otherwise we could be stuck trying new ideas all day.... Use lotsOfBitsIf needed don't forget to include songs written either solo OR live recording here . If anything fails please let us see how much effort has been put into creating these pieces themselves because no matter whether anyone else agrees with them whatsoever , nobody wants someone getting paid off after working through his process until he gets enough money back!!! Thanks again!!


FINE-TUNED GPT:
Light light walk come like wind hear talk,
Moon black dark night white sky gray look,
Sleepy tell lie dream come awake,
Dance play light talk move go back walk,


--- TEST CASE ---
Chosen topic: 2 - romantic
Chosen keywords: ['1 - timeless', '7 - happy', '8 - energetic']

The prompt: Generate a line for a blues song about world/life with timeless and happy and energetic feel.


GPT SMALL RAW:
This is not the first time we have created this project, so be sure to check it out! We hope you enjoy our work as much as us…and please share your thoughts in comments below or on social media using #myknees

FINE-TUNED GPT:
Little little boy girl think go,
Like forgive blame stay lose smile show,
Forget take change come home break go,
Inside head cold sweat warm feet grow,
"""

## Human evaluation

In [ ]:
"""
Linkert scale:
0: Stongly disagree
1: Disagree
2: Neither disagree nor agree
3: Agree
4: Stongly agree

Characteristics: coherence, creativity, rhymre
"""

In [ ]:
"""
--- TEST CASE ---
Chosen topic: 0 - romantic
Chosen keywords: ['6 - danceable', '8 - energetic']

FINE-TUNED GPT:
Like like tell give go know come let,
Sky go green white baby girl heart,
Go stand hear wonder leave stare night,
Stand come back home look say goodbye wait,

VOLUNTEER I:
Coherence:2
Creativity:3
Topicality:1
VOLUNTEER II:
Coherence:1
Creativity:2
Topicality:3

--- TEST CASE ---
Chosen topic: 1 - romantic
Chosen keywords: ['0 - violent']

FINE-TUNED GPT:
Come come stay away cry go tell,
Good goodbye nice big happy little,
Night time walk sleep come play talk tell,
Get somethin' come fall time ****,

VOLUNTEER I:
Coherence: 1
Creativity: 2
Topicality: 2
VOLUNTEER II:
Coherence: 1
Creativity: 2
Topicality: 2

--- TEST CASE ---
Chosen topic: 2 - romantic
Chosen keywords: ['1 - timeless', '2 - temporal']

FINE-TUNED GPT:
Love come love fall look see eye tell,
Like love stay hold leave mind wait till,
Love kiss goodbye hug heart walk leave soul,
Sleep know give say come out stay tell,

VOLUNTEER I:
Coherence:3
Creativity:3
Topicality:4
VOLUNTEER II:
Coherence:2
Creativity:4
Topicality:3

--- TEST CASE ---
Chosen topic: 3 - romantic
Chosen keywords: ['4 - sad', '5 - emotional']

FINE-TUNED GPT:
Good morning heart good night smile hand,
Give remember say life keep hold,
Rain drop shine fall sunlight touch wind,
Think say goodbye love stop pain talk mind,

VOLUNTEER I:
Coherence:4
Creativity:3
Topicality:4
VOLUNTEER II:
Coherence:3
Creativity:3
Topicality:4

--- TEST CASE ---
Chosen topic: 4 - romantic
Chosen keywords: ['3 - romantic', '5 - emotional']

FINE-TUNED GPT:
Lose stay lose sleep talk like leave kiss,
Just try again next year sweet miss,
Come go get baby see look face,
Remember all time happy dance,

VOLUNTEER I:
Coherence:3
Creativity:3
Topicality:4
VOLUNTEER II:
Coherence:4
Creativity:3
Topicality:4

--- TEST CASE ---
Chosen topic: 5 - romantic
Chosen keywords: ['5 - emotional', '7 - happy']

FINE-TUNED GPT:
Blue sky blue heaven earth soul,
Go up come out stay down look tell,
Go want come right away hear soul,
Cause cause wait long come die lie fall,

VOLUNTEER I:
Coherence:2
Creativity:3
Topicality:2
VOLUNTEER II:
Coherence:1
Creativity:2
Topicality:2

--- TEST CASE ---
Chosen topic: 6 - romantic
Chosen keywords: ['1 - timeless']

FINE-TUNED GPT:
Cold come get cold go give way blow,
Want see wonder forget dream know,
Dreamin dreamnin dreams mind heart go,
Hope dream wish make sure tomorrow,

VOLUNTEER I:
Coherence:3
Creativity:4
Topicality:3
VOLUNTEER II:
Coherence:2
Creativity:4
Topicality:2

--- TEST CASE ---
Chosen topic: 0 - romantic
Chosen keywords: ['8 - energetic']

FINE-TUNED GPT:
Light dance light look go come out shine,
Cry call do die young tears like pain,
Go to sleep baby break open,
Stay long walk lonely night listen,

VOLUNTEER I:
Coherence: 1
Creativity: 2
Topicality: 2
VOLUNTEER II:
Coherence: 2
Creativity: 2
Topicality: 2

--- TEST CASE ---
Chosen topic: 3 - romantic
Chosen keywords: ['4 - sad']

FINE-TUNED GPT:
Light light walk come like wind hear talk,
Moon black dark night white sky gray look,
Sleepy tell lie dream come awake,
Dance play light talk move go back walk,

VOLUNTEER I:
Coherence: 2
Creativity: 3
Topicality: 2
VOLUNTEER II:
Coherence: 2
Creativity: 2
Topicality: 2

--- TEST CASE ---
Chosen topic: 2 - romantic
Chosen keywords: ['1 - timeless', '7 - happy', '8 - energetic']

FINE-TUNED GPT:
Little little boy girl think go,
Like forgive blame stay lose smile show,
Forget take change come home break go,
Inside head cold sweat warm feet grow,

VOLUNTEER I:
Coherence: 3
Creativity: 3
Topicality: 2
VOLUNTEER II:
Coherence: 3
Creativity: 3
Topicality: 2

Average human evaluation scores:
Coherence: 2.25 / 5
Creativity: 2.80 / 5
Topicality: 2.60 / 5

"""

## Fine-tuned vs Gemini

In [ ]:
"""
--- TEST CASE ---
Chosen topic: 0 - romantic
Chosen keywords: ['6 - danceable', '8 - energetic']

The prompt: Generate a line for a blues song about music with danceable and energetic feel.

FINE-TUNED GPT:
Like like tell give go know come let,
Sky go green white baby girl heart,
Go stand hear wonder leave stare night,
Stand come back home look say goodbye wait,

GEMINI:
Here are a few options for a blues song line about music with a danceable and energetic feel:

* "The rhythm hits me deep inside, makes my feet wanna glide."
* "Got that backbeat thumpin' low, can't help but lose control."
* "This guitar wails, the drums ignite, gonna dance away the night."


--- TEST CASE ---
Chosen topic: 1 - romantic
Chosen keywords: ['0 - violent']

The prompt: Generate a line for a blues song about violence with violent feel.

FINE-TUNED GPT:
Come come stay away cry go tell,
Good goodbye nice big happy little,
Night time walk sleep come play talk tell,
Get somethin' come fall time ****,

GEMINI:
Please note: The following line aims to evoke a sense of unease and the harsh reality of violence, as is common in the blues genre, without being gratuitous or promoting actual harm.

Here's a line for a blues song about violence:

"Pistol smoke still hangin' low, another good man gone for show."


--- TEST CASE ---
Chosen topic: 2 - romantic
Chosen keywords: ['1 - timeless', '2 - temporal']

The prompt: Generate a line for a blues song about world/life with timeless and temporal feel.

FINE-TUNED GPT:
Love come love fall look see eye tell,
Like love stay hold leave mind wait till,
Love kiss goodbye hug heart walk leave soul,
Sleep know give say come out stay tell,

GEMINI:
Here's a line for a blues song about the timeless and temporal nature of the world/life:

"Sun keeps risin', night keeps fallin', same old song, just a different callin'."


--- TEST CASE ---
Chosen topic: 3 - romantic
Chosen keywords: ['4 - sad', '5 - emotional']

The prompt: Generate a line for a blues song about sadness with sad and emotional feel.

FINE-TUNED GPT:
Good morning heart good night smile hand,
Give remember say life keep hold,
Rain drop shine fall sunlight touch wind,
Think say goodbye love stop pain talk mind,

GEMINI:
Here's a line for a blues song about sadness:

"Tears like rain on a dusty pane, washin' 'round my heart's deep pain."


--- TEST CASE ---
Chosen topic: 4 - romantic
Chosen keywords: ['3 - romantic', '5 - emotional']

The prompt: Generate a line for a blues song about a romantic topic with romantic and emotional feel.

FINE-TUNED GPT:
Lose stay lose sleep talk like leave kiss,
Just try again next year sweet miss,
Come go get baby see look face,
Remember all time happy dance,

GEMINI:
"My love for you is a sweet, sweet blues, a melody etched deep within my soul."


--- TEST CASE ---
Chosen topic: 5 - romantic
Chosen keywords: ['5 - emotional', '7 - happy']

The prompt: Generate a line for a blues song about feelings with emotional and happy feel.

FINE-TUNED GPT:
Blue sky blue heaven earth soul,
Go up come out stay down look tell,
Go want come right away hear soul,
Cause cause wait long come die lie fall,

GEMINI:
"Sunrise paints the window pane, just like your love lights up my soul."


--- TEST CASE ---
Chosen topic: 6 - romantic
Chosen keywords: ['1 - timeless']

The prompt: Generate a line for a blues song about night/time with timeless feel.

FINE-TUNED GPT:
Cold come get cold go give way blow,
Want see wonder forget dream know,
Dreamin dreamnin dreams mind heart go,
Hope dream wish make sure tomorrow,

GEMINI:
"When the moon hangs high and the stars ignite, time just fades into the endless night."


--- TEST CASE ---
Chosen topic: 0 - romantic
Chosen keywords: ['8 - energetic']

The prompt: Generate a line for a blues song about music with energetic feel.

FINE-TUNED GPT:
Light dance light look go come out shine,
Cry call do die young tears like pain,
Go to sleep baby break open,
Stay long walk lonely night listen,

GEMINI:
"When that guitar cries and the drums just pound, ain't no stopping this feeling I've found!"


--- TEST CASE ---
Chosen topic: 3 - romantic
Chosen keywords: ['4 - sad']

The prompt: Generate a line for a blues song about sadness with sad feel.

FINE-TUNED GPT:
Light light walk come like wind hear talk,
Moon black dark night white sky gray look,
Sleepy tell lie dream come awake,
Dance play light talk move go back walk,

GEMINI:
"Every tear I cry just falls like rain, washing over this never-ending pain."


--- TEST CASE ---
Chosen topic: 2 - romantic
Chosen keywords: ['1 - timeless', '7 - happy', '8 - energetic']

The prompt: Generate a line for a blues song about world/life with timeless and happy and energetic feel.

FINE-TUNED GPT:
Little little boy girl think go,
Like forgive blame stay lose smile show,
Forget take change come home break go,
Inside head cold sweat warm feet grow,

GEMINI:
"This ol' world keeps spinnin' 'round, and every day's a joyful sound!"
"""